# Lecture 6 Homework — Build a Multi-Agent Research Assistant

**Student starter notebook (not a solution)**

### Your mission
Build a research assistant that can:
1. **Coordinate multiple agents** to answer a research question.
2. **Remember previous messages across restarts/sessions** using persistent storage.
3. **Handle failures** with appropriate retry logic and clear fallbacks.
4. **Explain your architecture choices** and trade-offs.

**Suggested difficulty:** core requirements first; optional extensions for advanced students.  
**Deliverables:** your completed `.ipynb`, a short architecture diagram/description, and evidence of tests (output or screenshots).

> The provided mock research data makes the homework possible without web-search access. You may integrate a real search API as an extension. Do not present mock data as verified live research.

## Architecture to start from

```text
User question
     |
     v
Supervisor (routes task)
     |-------------------------|
     v                         v
Research agent             Writer agent
(collect evidence)         (form answer from evidence)
     |                         |
     +---- shared graph state -+
                    |
                    v
              Final response

Persistent checkpoint store (SQLite) saves state per thread_id.
```

This is a starting point, not the only acceptable architecture. **Core goal:** supervisor routes work to at least two specialist nodes; researcher gathers evidence, writer generates a sourced answer, and the result is saved under a persistent conversation thread.

Suggested flow: `START → supervisor → research → writer → END`. A more dynamic supervisor that chooses the next agent at each step is an optional enhancement.

**Design note:** A research agent does not automatically need a separate model. A node with a retrieval tool is acceptable if you explain your choice.

## 0. Setup

Install into your notebook environment (uncomment if needed), then restart the kernel after installation. **Python 3.11+ recommended.**

For OpenAI-compatible classroom deployments, initialize a model with the provider settings your instructor gave you. The helper below uses `ChatOpenAI` and accepts an optional compatible base URL. For Azure-specific deployments that require an API version, use the client configuration from your class instead.

In [ ]:
# %pip install -U langgraph langgraph-checkpoint-sqlite langchain-openai tenacity


In [ ]:
import os
import sqlite3
from pathlib import Path
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# The notebook also runs with mock research data, but the writer uses an LLM.
# Set OPENAI_API_KEY (and optionally OPENAI_MODEL / OPENAI_ENDPOINT) in your environment.
from langchain_openai import ChatOpenAI

# if os.getenv("OPENAI_MODEL") doesn't work, cehck if it's not called DEPLOYMENT in your .env.
llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_ENDPOINT"),
    temperature=0,
)


## 1. Build the shared graph state

**TODO 1:** Choose which fields your agents need. A starter schema is below. Add fields if necessary.

- `question` is the user's request.
- `evidence` is retrieved material.
- `answer` is the final response.
- `next_agent` is optional if you use conditional routing.

Keep the flow easy to debug; don't add state variables only because they seem advanced.

In [ ]:
class ResearchState(TypedDict):
    question: str
    evidence: list[str]
    answer: str
    # TODO: add any additional fields required by your architecture


## 2. Research material and specialist nodes

The lookup function below provides **simulated** information so everyone can test without a paid search service. Replace it with a real tool only if you wish.

**TODO 2:** Implement `research_agent` to call `search_mock_knowledge` and return evidence to state.  
**TODO 3:** Implement `writer_agent` to answer only from the evidence, and to say when evidence is insufficient.

In [ ]:
KNOWLEDGE_BASE = {
    "langgraph": [
        "LangGraph models workflows as stateful graphs of nodes and edges.",
        "A checkpointer can persist graph state for a conversation thread.",
    ],
    "memory": [
        "Thread-scoped checkpoints support continuing the same conversation.",
        "Cross-thread user preferences require a separately managed store or database.",
    ],
    "multi-agent": [
        "Multi-agent systems can separate research, writing, and review responsibilities.",
        "Additional agents introduce coordination overhead, cost, and more failure points.",
    ],
}

def search_mock_knowledge(query: str) -> list[str]:
    """Offline demonstration data; not a live fact-checking/search engine."""
    query = query.lower()
    findings = []
    for keyword, facts in KNOWLEDGE_BASE.items():
        if keyword in query:
            findings.extend(facts)
    return findings

# Sanity check:
print(search_mock_knowledge("How does LangGraph memory work?"))


In [ ]:
def research_agent(state: ResearchState) -> dict:
    # TODO: use search_mock_knowledge with state["question"]
    # Return a PARTIAL state update such as {"evidence": ...}
    raise NotImplementedError("TODO 2: implement research_agent")


def writer_agent(state: ResearchState) -> dict:
    # TODO: construct a prompt using the question and evidence
    # Invoke llm and return {"answer": response.content}
    # IMPORTANT: if there is no evidence, do not invent citations or facts.
    raise NotImplementedError("TODO 3: implement writer_agent")


### Optional: Add a reviewer

The reviewer can check whether the draft is grounded in evidence and request one revision. Avoid an unlimited critique loop; include a maximum iteration limit if you implement feedback.

## 3. Add the supervisor and graph routing

**TODO 4:** Implement a supervisor and wire up the graph. For the **minimum acceptable implementation**, the supervisor can route to research first and writing second using a fixed sequence. For a stronger solution, use an LLM with structured output to choose between `research`, `writer`, and `END`.

Check that every possible branch terminates. You must have at least two specialist nodes.

In [ ]:
def supervisor_node(state: ResearchState) -> dict:
    # TODO: decide routing. For a basic sequence, this node may only
    # initialize metadata, followed by a fixed edge to 'research'.
    return {}

builder = StateGraph(ResearchState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("research", research_agent)
builder.add_node("writer", writer_agent)

builder.add_edge(START, "supervisor")
# TODO 4: add edges (and optionally conditional edges)
# HINT: baseline workflow: supervisor -> research -> writer -> END


## 4. Persist conversation state using SQLite

`InMemorySaver` is lost when Python stops. Here, SQLite checkpointing writes to disk. **TODO 5:** Compile your graph with the checkpointer, run it, then restart the kernel and rerun setup against the *same SQLite file and `thread_id`*.

**Important distinction:** A checkpoint is thread-scoped. It is not automatically a user-wide memory system. Reusing the same thread ID continues that thread; different thread IDs remain separate.

SQLite holds a connection open below for the current notebook session. Re-running the setup after a restart reopens the existing file.

In [ ]:
db_path = Path("lecture6_research_checkpoints.sqlite")
conn = sqlite3.connect(str(db_path), check_same_thread=False)
checkpointer = SqliteSaver(conn)

# TODO 5: finish graph routing above before compiling
# app = builder.compile(checkpointer=checkpointer)

thread_a = {"configurable": {"thread_id": "research-session-A"}}
thread_b = {"configurable": {"thread_id": "research-session-B"}}

# TODO: invoke your graph on a question using thread_a.
# TODO: restart kernel and reuse same SQLite file and thread_a.
# TODO: compare checkpointed state with thread_b.


### Optional: Remember preferences across separate conversations

A checkpointed thread is *not* the same as durable user memory shared across threads. For an extension, save a simple profile (`user_id`, `preferred_style`) in a separate SQLite table and retrieve it from a second conversation. The skeleton below shows the table only.

In [ ]:
preferences_db = sqlite3.connect("lecture6_user_preferences.sqlite")
preferences_db.execute("""
CREATE TABLE IF NOT EXISTS preferences (
    user_id TEXT PRIMARY KEY,
    preferred_style TEXT NOT NULL
)
""")
preferences_db.commit()

# OPTIONAL TODO: add functions save_preference(user_id, style)
# and read_preference(user_id). Use bound SQL parameters (?) rather than string formatting.


## 5. Add error handling and retries

**TODO 6:** Use retries for *transient* errors (timeouts or temporary rate limits), but not for permanent errors (bad credentials or invalid input). Show what your application does after the maximum number of retries.

A simulated exception below is used for offline testing. Replace or supplement it when integrating a real provider.

In [ ]:
class TemporarySearchError(Exception):
    """Simulated transient tool outage."""

class InvalidSearchRequest(Exception):
    """Simulated permanent input error."""

attempts = {"count": 0}

def flaky_lookup(query: str) -> list[str]:
    if not query.strip():
        raise InvalidSearchRequest("Query cannot be empty")
    attempts["count"] += 1
    if attempts["count"] <= 2:
        raise TemporarySearchError("Temporary service failure")
    return search_mock_knowledge(query)

# TODO 6: Write a @retry decorator (or retry wrapper) that retries
# TemporarySearchError at most 3 times with exponential backoff.
# Do NOT retry InvalidSearchRequest.
# TODO: decide a fallback when all retries fail.


## 6. Test your solution

Run the checks below **after** implementing the TODOs. The assertions assume your `app` graph returns an `answer` key and uses `evidence` from the research node. Adjust only if your design intentionally uses another schema.

### Core test checklist

- [ ] Supervisor invokes at least two distinct specialist nodes.
- [ ] Research agent collects evidence; writer uses it to answer.
- [ ] Unknown-topic query produces a clear uncertainty statement rather than invented evidence.
- [ ] You can query or continue a conversation using a saved SQLite thread after kernel restart.
- [ ] Two thread IDs remain separate.
- [ ] A temporary error is retried; permanent errors are not.
- [ ] Max retries lead to a controlled error message or fallback.
- [ ] You can explain one alternative architecture and why you did not choose it.

In [ ]:
# Uncomment after completing the implementation:
# response = app.invoke(
#     {"question": "Explain LangGraph memory", "evidence": [], "answer": ""},
#     thread_a,
# )
# assert isinstance(response["answer"], str) and response["answer"].strip()
# assert len(response["evidence"]) > 0
# print(response["answer"])


## 7. Document your architecture decisions (required)

Write about **200–400 words** or a small diagram plus bullet points. Complete these prompts:

1. **Architecture:** What agents/nodes exist, and what does each one do?
2. **Routing:** Why did you choose fixed sequence, supervisor routing, or another design?
3. **Memory:** What exactly is stored, where, and how do you resume it after a restart? Is it thread-specific or cross-thread?
4. **Errors:** Which errors are retried, how many times, and what is the fallback?
5. **Trade-offs:** What does the multi-agent design gain and cost relative to one agent with tools?
6. **Limitations:** What can the assistant *not* reliably do with the current data/tools?

**Your answer:**

> TODO: Replace this with your architecture notes.


## Optional challenges (choose one)

- **Reviewer agent:** Checks the answer against the retrieved evidence and requests one revision if necessary.
- **Dynamic supervisor:** Uses structured output for constrained routing decisions.
- **Live research:** Replace mock lookup with a permitted web-search tool; include source URLs and handle timeouts.
- **Cross-thread preference memory:** Save a user's preferred response style in durable storage and retrieve it in a separate thread.
- **Human-in-the-loop:** Require approval before sending a report externally (simulate sending; do not send real messages).

### Submission reminders
Save your completed notebook with relevant outputs, include your architecture notes, and make sure no API keys or private student data are embedded in the file.
